Docker is a platform for packaging and running applications inside **containers** — lightweight, portable units that bundle your code, runtime, dependencies, and configuration into a single artifact. A container runs the same way on a developer's laptop, a CI server, and a production cloud instance.

In this notebook we'll cover:
- The problem Docker was built to solve
- What a container actually is under the hood
- How containers compare to virtual machines
- Docker's architecture and core concepts
- Running your very first container

## The Problem: "It Works on My Machine"

Imagine you build a web app on macOS with Python 3.11. It works perfectly. You hand it to a colleague running Ubuntu 20.04 with Python 3.8 — and it fails immediately. A library version differs, a system dependency is missing, an environment variable isn't set.

This is the **"works on my machine" problem**, and it has plagued software teams for decades. The environment the app was *built* in is different from the environment it *runs* in.

Before containers, teams reached for two solutions:

| Approach | How it works | The catch |
|---|---|---|
| **Setup scripts** | A shell script installs the right versions on any machine | Scripts drift over time; different OSes behave differently |
| **Virtual Machines** | Ship a full OS image with your app baked in | Heavy (GBs per VM), slow to start, wasteful on shared hardware |

Docker's answer: package **everything** the application needs — code, runtime, libraries, config — into one image that runs identically anywhere Docker is installed. Ship the environment, not just the code.

## What is a Container?

A container is an **isolated process** running on the host operating system's kernel, with its own filesystem, network stack, and process namespace — but without the overhead of a full virtual machine.

It achieves this isolation using two Linux kernel features:

**Namespaces** — partition kernel resources so each container sees only its own slice of the world:
- `pid` namespace: the container has its own process tree. PID 1 inside the container is your app, not the host's init.
- `net` namespace: the container has its own network interfaces and routing table.
- `mnt` namespace: the container has its own filesystem view.
- `uts` namespace: the container can have its own hostname.

**cgroups (control groups)** — enforce resource *limits*: how much CPU, RAM, and disk I/O a container is allowed to consume.

Together, namespaces give isolation and cgroups give resource control. Docker wraps these kernel primitives in a user-friendly CLI and image format — that is the core of what Docker *is*.

## Virtual Machines vs Containers

Both VMs and containers solve the environment problem, but at very different levels of the stack.

```
Virtual Machine stack          Container stack
─────────────────────          ───────────────────
  App                            App
  Runtime / Libs                 Runtime / Libs
  Guest OS (full kernel)         (no guest OS)
  Hypervisor                     Docker Engine
  Host OS                        Host OS
  Hardware                       Hardware
```

A VM carries an entire operating system — kernel, init system, system libraries — for every workload. A container shares the host kernel and only bundles what the application actually needs.

| | Virtual Machine | Container |
|---|---|---|
| **OS** | Full guest OS per VM | Shares host kernel |
| **Size** | GBs per image | MBs per image |
| **Startup time** | Minutes | Milliseconds to seconds |
| **Isolation** | Strong (separate kernel) | Process-level (shared kernel) |
| **Density** | Tens per host | Hundreds per host |
| **Portability** | VM format tied to hypervisor | Docker image runs anywhere |

> Containers and VMs are not mutually exclusive. In production, containers often *run inside* VMs — you get the hardware isolation of a VM and the density and speed of containers.

## What is Docker?

Docker is an open-source platform that makes containers easy to build, ship, and run. It was created by Solomon Hykes and released publicly in March 2013 by a company then called dotCloud (later renamed Docker, Inc.).

Before Docker, Linux containers existed (LXC since 2008), but they were difficult to use — no standard image format, no easy way to share containers, no simple CLI. Docker provided:

| What Docker added | Why it mattered |
|---|---|
| **Dockerfile** — a declarative build script | Reproducible, version-controlled image builds |
| **Layered image format** | Efficient storage; shared layers between images |
| **Docker Hub** — a public registry | Share and pull images with one command |
| **Simple CLI** (`docker run`, `docker build`) | No kernel expertise required |

Docker went from zero to 100 million downloads in under a year. It fundamentally changed how software is deployed and spawned an entire ecosystem — Kubernetes, container registries, CI/CD pipelines — that is now the backbone of modern cloud infrastructure.

| Year | Milestone |
|---|---|
| **2008** | Linux Containers (LXC) introduced in the kernel |
| **2013** | Docker 0.1 released publicly at PyCon |
| **2014** | Docker 1.0 — production-ready; Docker Hub launched |
| **2015** | Docker Compose, Docker Swarm; OCI standard formed |
| **2017** | Kubernetes becomes the dominant orchestrator |
| **2019** | Docker Desktop for Mac/Windows widely adopted |
| **2020+** | containerd (Docker's runtime) donated to CNCF; Docker Engine powers most cloud platforms |

## Docker Architecture

Docker uses a **client–server architecture**. The CLI you type commands into is the *client*; the heavy lifting is done by a background daemon called `dockerd`.

```
┌──────────────────────────────────────────────────────────┐
│  Your terminal                                           │
│  docker build / docker run / docker pull                 │
│              │  (REST API over Unix socket or TCP)       │
│              ▼                                           │
│  ┌─────────────────────┐      ┌───────────────────────┐ │
│  │   Docker Daemon     │◄────►│   Docker Registry     │ │
│  │   (dockerd)         │      │   (Docker Hub /       │ │
│  │                     │      │    private registry)  │ │
│  │  ┌───────────────┐  │      └───────────────────────┘ │
│  │  │  containerd   │  │                                 │
│  │  │  (runtime)    │  │                                 │
│  │  └───────┬───────┘  │                                 │
│  │          │          │                                 │
│  │  ┌───────▼───────┐  │                                 │
│  │  │  Containers   │  │                                 │
│  │  └───────────────┘  │                                 │
│  └─────────────────────┘                                 │
└──────────────────────────────────────────────────────────┘
```

- **Docker Client** — the `docker` CLI. Translates your commands into API calls.
- **Docker Daemon (`dockerd`)** — listens for API requests, manages images, containers, networks, and volumes.
- **containerd** — the low-level runtime that actually creates and runs containers using kernel namespaces and cgroups. Docker donated it to the CNCF in 2017.
- **Docker Registry** — stores and distributes images. Docker Hub is the default public registry. You can also run private registries (AWS ECR, GCR, Harbor).

## Core Concepts

Four terms appear constantly in Docker. Understanding how they relate to each other is the foundation for everything else.

### Image
A **read-only template** that defines what is inside a container — the filesystem, environment variables, default command, and metadata. Think of it like a *class* in object-oriented programming: the blueprint, not the running thing.

Images are built in **layers**. Each instruction in a Dockerfile adds a layer. Layers are cached and shared across images — if two images both use `ubuntu:22.04` as a base, they share those layers on disk.

### Container
A **running instance of an image** — the class brought to life as an object. You can run many containers from the same image simultaneously. Each gets its own isolated filesystem, network, and process namespace, but they all start from the same immutable image.

### Dockerfile
A **plain-text recipe** for building an image. Each line is an instruction (`FROM`, `RUN`, `COPY`, `CMD`, etc.). Docker reads the Dockerfile top to bottom and produces a layered image.

```dockerfile
FROM python:3.11-slim          # start from an official base image
WORKDIR /app                   # set the working directory inside the container
COPY requirements.txt .        # copy files from host into the image
RUN pip install -r requirements.txt  # run a command during the build
COPY . .                       # copy the rest of the app
CMD ["python", "app.py"]       # default command when a container starts
```

### Registry
A **storage and distribution system for images**. You `docker push` to upload an image; `docker pull` to download one. Docker Hub (`hub.docker.com`) is the default public registry — it hosts official images for Python, Node.js, Nginx, PostgreSQL, and thousands more.

## Hands-On: Docker in Action

The code cells below run `docker` CLI commands. Make sure Docker Desktop (or Docker Engine on Linux) is running before executing them.

In [ ]:
# Confirm Docker is installed and the daemon is running
!docker version --format 'Client: {{.Client.Version}}  Server: {{.Server.Version}}'

In [ ]:
# The classic first container — Docker pulls the image, runs it, and exits
# hello-world is a minimal image that prints a confirmation message
!docker run --rm hello-world

In [ ]:
# Show images now stored locally (hello-world was pulled automatically)
!docker images

In [ ]:
# Run an Ubuntu container and print its OS release info
# --rm: remove the container after it exits
# Notice: this runs Ubuntu on top of your host OS with no VM — just a process
!docker run --rm ubuntu:22.04 cat /etc/os-release

In [ ]:
# Run an Alpine Linux container and print its OS release info
# Alpine is a 5 MB minimal image — compare that to a full Ubuntu VM (~1 GB)
!docker run --rm alpine:3.19 cat /etc/os-release

In [ ]:
# Run two different Python versions simultaneously — no version conflicts
# Each container is isolated; they don't know about each other
!docker run --rm python:3.9-slim python --version
!docker run --rm python:3.12-slim python --version

In [ ]:
# List all containers (including stopped ones)
# --rm above deleted containers after exit, so only manually stopped ones appear
!docker ps -a

In [ ]:
# Inspect image layer history — see how an image is built up in layers
# Each layer corresponds to one instruction in the Dockerfile
!docker history python:3.12-slim --no-trunc=false

## Summary

- The **"works on my machine" problem** arises when the build and runtime environments differ. Docker solves this by packaging the environment itself.
- A **container** is an isolated process on the host kernel, using Linux **namespaces** for isolation and **cgroups** for resource limits — no guest OS required.
- **Containers vs VMs**: containers share the host kernel (lightweight, millisecond startup, MBs); VMs carry a full OS (heavyweight, minute startup, GBs).
- **Docker** (released 2013) made containers practical by adding a layered image format, Dockerfile, Docker Hub, and a simple CLI.
- Docker's architecture: **Client** → **Daemon (`dockerd`)** → **containerd** → containers. Images are stored in a **Registry**.
- The four core concepts: **Image** (blueprint), **Container** (running instance), **Dockerfile** (build recipe), **Registry** (image storage).
- `docker run --rm hello-world` is the canonical first step — Docker pulls the image, runs it, and cleans up.